In [ ]:
from llama_index.core import SimpleDirectoryReader

In [ ]:
import sys
sys.path.append("..")
from src.helper import get_groq_api_key, get_nvidia_api_key, get_hf_token
import nest_asyncio

GROQ_API_KEY = get_groq_api_key()
NVIDIA_API_KEY = get_nvidia_api_key()
HF_TOKEN = get_hf_token()
nest_asyncio.apply()

d:\Building Agentic RAG with Llamaindex\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from src.helper import PDFPlumberReader

extractors_override = {
    ".pdf": PDFPlumberReader()
}

reader = SimpleDirectoryReader(
    input_files=["metagpt.pdf"],
    file_extractor=extractors_override
)

documents = reader.load_data()

In [4]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(
    documents,
    show_progress=True
)

Parsing nodes: 100%|██████████| 29/29 [00:06<00:00,  4.23it/s]


In [5]:
from llama_index.core import Settings
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.nvidia import NVIDIAEmbedding

Settings.llm = OpenAILike(
    api_key=HF_TOKEN,
    api_base="https://router.huggingface.co/v1",
    model="meta-llama/Llama-3.1-8B-Instruct",
    is_chat_model=True
)
Settings.embed_model = NVIDIAEmbedding(
    nvidia_api_key=NVIDIA_API_KEY,
    model="nvidia/nemotron-3-embed-1b"
)

d:\Building Agentic RAG with Llamaindex\.venv\Lib\site-packages\llama_index\embeddings\nvidia\base.py:206: UserWarning: Unable to determine validity of nvidia/nemotron-3-embed-1b
  warnings.warn(f"Unable to determine validity of {model_name}")


In [6]:
from llama_index.core import SummaryIndex, VectorStoreIndex

summary_index = SummaryIndex(
    nodes, 
    show_progress=True
)
vector_index = VectorStoreIndex(
    nodes,
    show_progress=True
)

Generating embeddings: 100%|██████████| 35/35 [00:06<00:00,  5.28it/s]


In [7]:
summary_index_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True
)
vector_index_engine = vector_index.as_query_engine()

In [8]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

summary_metadata = ToolMetadata(
    description="Useful for summarization questions related to MetaGPT",
    name="summary"
)

vector_metadata = ToolMetadata(
    description="Useful for retrieving specific context from the MetaGPT paper.",
    name="vector_store"
)


summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_index_engine,
    description=summary_metadata.description,
    name=summary_metadata.name,
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_index_engine,
    description=vector_metadata.description,
    name=vector_metadata.name,
)

In [9]:
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine

query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[summary_tool, vector_tool],
    verbose=True
)

In [10]:
response = query_engine.query("What is summary of MetaGPT?")
print(str(response))

Selecting query engine 0: The question is asking for a summary of MetaGPT, which suggests a high-level overview or explanation, making choice 1 more relevant for summarization questions..
MetaGPT is a framework that enables collaborative teamwork among agents to develop software projects. It allows for the creation of a software development team that can learn from past experiences and improve over time. The framework utilizes a specialized division of labor, Standard Operating Procedures (SOPs), and a self-referential mechanism to enhance code generation quality, achieve high performance in software development tasks, and ensure user data privacy and security.


In [12]:
response = query_engine.query("What is MetaGPT?")
print(str(response))

APIStatusError: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}